<a href="https://colab.research.google.com/github/GimenesPaula/GimenesPaula/blob/main/lanc_vendas_anos_Focus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bibliotecas Phyton

In [ ]:
!pip install requests

In [ ]:
pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.0 MB/s eta 0:00:00


In [ ]:
#Carrega Bibliotecas
import pandas as pd
import numpy as np
import re
from functools import lru_cache
from unidecode import unidecode
import requests

# Fazer Upload planilhas
1.   Lançamentos
2.   Inci
3.   Vendas

In [ ]:
#Lançamentos
from google.colab import files
uploaded = files.upload()

Saving Lançamentos Jan Abr 2026.xlsx to Lançamentos Jan Abr 2026.xlsx


In [ ]:
filename = next(iter(uploaded))

In [ ]:
# INCI name produtos
from google.colab import files
inci = files.upload()

Saving inci_16_08.xlsx to inci_16_08.xlsx


In [ ]:
filename2 = next(iter(inci))

In [ ]:
#Vendas Distribuidores
from google.colab import files
dist = files.upload()

Saving Focus Sales Report Q2 2026.xlsx to Focus Sales Report Q2 2026.xlsx


In [ ]:
filename4 = next(iter(dist))

# Análise Ferramenta de Vendas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data = '/content/'+filename4
df_sale = pd.read_excel(data)
pd.options.display.max_columns=None
df_sale.nunique()

,0
Distributor (subsidiary),1
WACKER Business Team Initials,1
WACKER Market Segment,0
WACKER Application,0
WACKER Material No.,63
WACKER Material Name,67
Customer No. (at distributor),0
Customer Name,1296
Corporate Group (of customer),0
Country code ship-to (of customer),1


### Gera chave para Fabricante

In [ ]:
#Dicionários
PALAVRAS_IRRELEVANTES = {
    'industria', 'comercio', 'cosmetico', 'cosmeticos', 'tecnologia', 'ltda', 'me', 'eireli', 'sa', 'cia', 'comercial', 'ind', 'e',
    'produtos', 'servicos', 'serviço', 'do', 'da', 'de', 'dos', 'das', 'the', 'group', 'grupo', 'laboratorio',
    'inc', 'corp', 'corporation', 'associados', 'associado', 'associacao', 'associação', 'holding', 'importadora',
    'exportadora', 'importacao', 'importação', 'exportacao', 'exportação', 'distribuidora', 'distribuidor', 'fabricacao',
    'fabricante', 'comerciante', 'comercio', 'comércio', 'comercial', 'empresa', 'sociedade', 'unipessoal',
    'aerosol', 'aerossol', 'technologies', 'prod', 'cosmetica', 'laboratorios',
    'atacado', 'quimica', 'industrial', 'pesquisas', 'cosmetic', 'beauty', 'higien', 'administradora',
    'fragrancias', 'aerossois', 'brasil', 'manufacturing', 'farmaceuticos', 'higiene', 'limpeza',
    'com', 'instituto', 'perfumes', 'epp', 'naturais', 'tercerizacao', 'cosmesticos', 'imp', 'natural',
    'higie', 'pessoal', 'quimicos', 'cosm', 'impo', 'para', 'lempeza', 'laboratorios,', 'farmacias',
}

In [ ]:
# Função para limpar texto
def limpar(texto):

    if pd.isnull(texto):
        return []

    texto = str(texto).strip()

    if texto.lower() in ['', 'nan', 'none', 'null']:
        return []

    texto = unidecode(texto).lower()
    texto = re.sub(r"[./\\’'-]", '', texto)
    texto = re.sub(r'\s+', ' ', texto)

    return [p for p in texto.split(' ') if p]

# Função para extrair chave representativa
def extrair_chave(palavras):
    if not palavras:
        return ''
    relevantes = [p for p in palavras if p not in PALAVRAS_IRRELEVANTES]
    if not relevantes:
        relevantes = palavras
    if len(relevantes) >= 3:
        return f"{relevantes[0]} {relevantes[1]} {relevantes[2]}"
    if len(relevantes) >= 2:
        return f"{relevantes[0]} {relevantes[1]}"
    else:
        chave = relevantes[0]
    chave = chave.strip()
    chave = re.sub(r'\b(e|and|&|\+)\b$', ' ', chave)
    chave = re.sub(r'\s+', ' ', chave)
    return chave

# Função para gerar chaves para DataFrame com fallback para valor bruto da coluna1
def gerar_chaves(df, coluna1, coluna2=None, coluna3=None):
    palavras1 = df[coluna1].apply(limpar)
    palavras2 = df[coluna2].apply(limpar) if coluna2 else pd.Series([[]] * len(df), index=df.index)
    palavras3 = df[coluna3].apply(limpar) if coluna3 else pd.Series([[]] * len(df), index=df.index)
    def chave_final(idx):
        for lista in [palavras1.iloc[idx], palavras2.iloc[idx], palavras3.iloc[idx]]:
            chave = extrair_chave(lista)
            if chave:
                return chave
        return str(df[coluna1].iloc[idx]).strip().lower()  # fallback para valor bruto da coluna1

    return pd.Series([chave_final(i) for i in range(len(df))], index=df.index)

### Gera a Chave de Material

In [ ]:
def formatar_material(coluna):
    def extrair_codigo(texto):
        #Tenta extrair o padrão: 2+ letras + 1+ números
        match = re.search(r'\b([A-Za-z]{2,}\s*\d{1,})\b', texto)
        if match:
            return match.group(1).upper().strip()
    return coluna.apply(extrair_codigo)

### Relatório de Vendas

In [ ]:
def resumo_por_ano(df, ano):
    df_ano = df[df['Year of Invoice'] == str(ano)]
    return (
        df_ano.groupby(['KeyManuf','Endereco','Region (State/Province) ship-to (of customer)'])
        .agg(
            **{f'Produtos_{ano}': ('WACKER Material Name', lambda x: len(set(i for i in x if i))),
               f'Volume_{ano}': ('Quantity actual period', 'sum'),
               f'Lista_{ano}': ('WACKER Material Name', lambda x: sorted(set(i for i in x if i)))}
        )
    )

In [ ]:
# Filtra apenas linhas que contenham 'BELSIL' na descrição (case-insensitive)
df_sale = df_sale[df_sale['WACKER Material Name'].str.contains('BELSIL', case=False, na=False)]

# Padroniza fabricantes e distribuidores
df_sale['KeyManuf'] = gerar_chaves(df_sale, 'Customer Name')

# Limpa e padroniza a coluna 'Material'
df_sale['WACKER Material Name'] = formatar_material(df_sale['WACKER Material Name'].astype(str))

# garante que o ano seja string
df_sale['Year of Invoice'] = df_sale['Year of Invoice'].astype(str)

df_sale['Endereco']='Brasil'

resumo_2025 = resumo_por_ano(df_sale, '2025')
resumo_2026 = resumo_por_ano(df_sale, '2026')

# Junta os resultados
df_final = pd.concat([resumo_2025, resumo_2026], axis=1).reset_index()

# Funções para itens adicionados e perdidos
def itens_adicionados(row):
    l2025 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    l2026 = set(row['Lista_2026']) if isinstance(row['Lista_2026'], list) else set()
    return sorted(l2026 - l2025)

def itens_perdidos(row):
    l2025 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    l2026 = set(row['Lista_2026']) if isinstance(row['Lista_2026'], list) else set()
    return sorted(l2025 - l2026)

# Função para unir todos os códigos vendidos (2023 e 2025)
def Itens_Vendidos(row):
    l2 = set(row['Lista_2025']) if isinstance(row['Lista_2025'], list) else set()
    l3 = set(row['Lista_2026']) if isinstance(row['Lista_2026'], list) else set()
    return sorted(l2 | l3)

# Cria as colunas de interesse
df_final['Itens_Adicionados'] = df_final.apply(itens_adicionados, axis=1)
df_final['Itens_Perdidos'] = df_final.apply(itens_perdidos, axis=1)
df_final['Itens_Vendidos_2025_2026'] = df_final.apply(Itens_Vendidos, axis=1)

df_final['Estado'] = df_final['Region (State/Province) ship-to (of customer)']
# Seleciona as colunas finais
df_final = df_final[
    ['KeyManuf', 'Endereco', 'Estado',
     'Produtos_2025', 'Produtos_2026',
     'Volume_2025', 'Volume_2026',
     'Itens_Adicionados', 'Itens_Perdidos',
     'Itens_Vendidos_2025_2026']
]

# Salva o resultado em Excel
df_final.to_excel('Vendas_BELSIL_por_ano.xlsx', index=False)
df_final.head()

,KeyManuf,Endereco,Estado,Produtos_2025,Produtos_2026,Volume_2025,Volume_2026,Itens_Adicionados,Itens_Perdidos,Itens_Vendidos_2025_2026
0,&co,Brasil,SP,3.0,2.0,419.5,36.0,[],[DM 5],"[DM 5, EG 6000, TMS 803]"
1,3fa,Brasil,GO,3.0,2.0,581.0,1440.0,[],[EG 5],"[EG 5, GB 1020, OW 2100]"
2,a & d,Brasil,SC,6.0,4.0,2487.0,365.0,[ADM 8105],"[ADM 8301, CM 740, TMS 803]","[ADM 8105, ADM 8301, ADM 9000, CM 740, DM 6010..."
3,a&s,Brasil,SP,4.0,3.0,5274.0,6142.0,[],[OW 2100],"[DM 0, DM 6010, OW 2100, TMS 803]"
4,abalo,Brasil,SP,6.0,1.0,60.0,20.0,[],"[ADM 9000, DM 6010, GB 3024, GB 3025, OW 2100]","[ADM 9000, DM 350, DM 6010, GB 3024, GB 3025, ..."


# Análise Relatórios Lançamentos

## Descritivo Lançamentos





In [ ]:
#Carrega o banco de dados como tabela
data = '/content/'+filename
df_launches = pd.read_excel(data)
pd.options.display.max_columns = None
df_launches.nunique()

,0
Número do Produto,2998
Data de Publicação,124
Produto,2114
Marca,1867
Empresa,683
Categoria,3
Sub-Categoria,33
Descrição do Produto,2992
Preço por 100g/ml,2205
Preço em moeda local,1203


In [ ]:
#Edita Coluna Ano
df_launches['Ano'] = pd.to_datetime(df_launches['Data de Publicação']).dt.year

##Upload INCI

In [ ]:
data = '/content/'+filename2
df_inci = pd.read_excel(data)
pd.options.display.max_columns=None
df_inci.nunique()

,0
Produto,51
Ingrediente,50
Prioridade,3


## Função Analisa Ingredientes

In [ ]:
#this checks if any combination of INCI as present in Ingredient
def verifica_ingrediente(formula,produto,material):
  quantidade = len(produto.difference(formula))
  if quantidade == 0:
    return material
  return None

In [ ]:
#If last code is true, this returns the Descrição name
def procura_produtos(formula, produtos, materiais):
  formula = formula.copy()
  resultados = []
  for prod, mat in zip(produtos, materiais):
    resultado = verifica_ingrediente(formula, prod, mat)
    if resultado is not None:
      formula = formula.difference(prod) ## Para remover os ingredientes já encontrados numa nova busca.
      resultados.append(resultado)
  return resultados

In [ ]:
# Função para sinalizar ingredientes do dictOTHERS
def sinaliza_ingredientes(x):
    ingredientes = set().union(*x)  # une todos os sets/listas de ingredientes do grupo
    encontrados = set()
    for ing in ingredientes:
        for palavra in dictOTHERS:
            if palavra.lower() in ing.lower():
                encontrados.add(ing)
    return ', '.join(sorted(encontrados))

## Função Transpoe coluna

In [ ]:
#this code transpose data. Used when we bring each category and the number of lauches.
def transpor (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:[])

In [ ]:
def to_set(x):
    s = set()
    for item in x:
        if isinstance(item, list):
            s.update(item)
        elif isinstance(item, str):
            s.add(item)
    return sorted(s)

In [ ]:
def transpor_2 (df, coluna, linha):
  for cat in df[coluna].unique():
    f = df[coluna] == cat
    df[cat] = df[f][linha]
    f = df[cat].isna()
    df.loc[f, cat] = df.loc[f, cat].apply(lambda x:x)

## Contém Silicone?

In [ ]:
#Dicionário Silicones geral
dictOTHERS = {'methicone':'1', 'Dimethicone':'1','methicone Crosspolymer':'1','methiconol':'1',
              'ylsiloxysilicate':'1', 'ylsilsesquioxane':'1', 'Disiloxane':'1', 'Silica':'1','siloxane':'1'}

In [ ]:
df_launches['Silicone'] = df_launches['Ingredients (Standard form)'].str.extract('('+'|'.join(dictOTHERS)+')',expand=False).map(dictOTHERS)

## Gera chave de Material


In [ ]:
#Edita tabela INCI

df_inci.dropna(inplace=True)
df_inci['Produto'] = formatar_material(df_inci['Produto'])

#Cria uma lista iterável dos ingredientes nos Produtos
df_inci['Ing'] = df_inci['Ingrediente'].str.split(', ').apply(set)
df_inci.sort_values('Prioridade', ascending=True, inplace=True)

In [ ]:
#Dicionário de palavras a remover da coluna Ingredientes no Mintel
dictIng = {
    r'\s*and/or\s*': ',',
    r',\s*': ',',
    r'\s*,': ',',
    r'\s*\(and\)\s*': ','
}

In [ ]:
# Cria coluna com produtos identificados
df_launches['Ingredients (Standard form)'] = df_launches['Ingredients (Standard form)'].astype(str)
#Cria uma lista iterável das ingredientes cosmeticos
df_launches['Ing'] = (
    df_launches['Ingredients (Standard form)']
    .replace(dictIng, regex=True)
    .str.split(',')
    .apply(set)
)

# Relaciona os ingredientes cosméticos
df_launches['Lançamentos'] = df_launches['Ing'].apply(
    lambda formulacao: procura_produtos(formulacao, df_inci['Ing'], df_inci['Produto'])
)

## Relatório de Lançamentos

In [ ]:
df_launches['Fabricante']= df_launches['Fabricante'].astype(str)
df_launches['Marca']= df_launches['Marca'].astype(str)
df_launches['Empresa']= df_launches['Empresa'].astype(str)
df_launches['KeyManuf'] = gerar_chaves(df_launches, 'Fabricante', 'Empresa', 'Marca')
df_launches['KeyMarca'] = gerar_chaves(df_launches, 'Marca')
df_launches['KeyEmpresa'] = gerar_chaves(df_launches, 'Empresa')

In [ ]:
# Dicionário de siglas e nomes de estados
estados = {
    'AC': 'Acre', 'AL': 'Alagoas', 'AP': 'Amapá', 'AM': 'Amazonas', 'BA': 'Bahia', 'CE': 'Ceará',
    'DF': 'Distrito Federal', 'ES': 'Espírito Santo', 'GO': 'Goiás', 'MA': 'Maranhão', 'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul', 'MG': 'Minas Gerais', 'PA': 'Pará', 'PB': 'Paraíba', 'PR': 'Paraná',
    'PE': 'Pernambuco', 'PI': 'Piauí', 'RJ': 'Rio de Janeiro', 'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul', 'RO': 'Rondônia', 'RR': 'Roraima', 'SC': 'Santa Catarina',
    'SP': 'São Paulo', 'SE': 'Sergipe', 'TO': 'Tocantins'
}

def extrair_estado(texto):
    if pd.isnull(texto):
        return None
    texto = str(texto).strip().lower()
    # Procura por sigla
    for sigla in estados:
        if re.search(r'\b' + re.escape(sigla.lower()) + r'\b', texto):
            return sigla
    # Procura por nome do estado
    for nome, sigla in estados_nome_para_sigla.items():
        if nome in texto:
            return sigla
    return None
# Inverte para buscar por nome também
estados_nome_para_sigla = {v.lower(): k for k, v in estados.items()}


# Exemplo: supondo que sua coluna de empresa é 'Fabricante'
df_launches['Estado'] = df_launches['Manufacturer Company Address'].apply(extrair_estado)

# Preencher os valores ausentes de 'Estado' com base em 'KeyManuf'
missing_estado = df_launches[df_launches['Estado'].isna()]
for index, row in missing_estado.iterrows():
    keymanuf = row['KeyManuf']

    # Buscar registros com o mesmo 'KeyManuf' e 'Estado' não nulo
    estado_valido = df_launches[
        (df_launches['KeyManuf'] == keymanuf) &
        (df_launches['Estado'].notna())
    ]['Estado'].unique()

    # Se houver apenas um valor único de 'Estado', preencher
    if len(estado_valido) == 1:
        df_launches.at[index, 'Estado'] = estado_valido[0]

In [ ]:
enderecos_por_empresa = df_launches.dropna(subset=['Local de fabricação']).groupby('KeyManuf')['Local de fabricação'].first().to_dict()

# Passo 2: Preencher os endereços faltantes com base no dicionário
df_launches['Endereco'] = df_launches.apply(lambda row: enderecos_por_empresa.get(row['KeyManuf'], row['Local de fabricação']), axis=1)


In [ ]:
df_launches['indice']=1
transpor_2(df_launches, 'Categoria', 'indice')

In [ ]:
#renomeia coluna categoria
df_launches.rename(columns={'Produtos para Pele':'Pele', 'Produtos para Cabelos':'Cabelos',
                            'Maquilagem': 'Make',
                            'Número do Produto':'Total Lançamentos'},inplace=True)

In [ ]:
df_c = df_launches.groupby(['KeyManuf', 'Endereco', 'Estado'], dropna=False)[['Pele', 'Cabelos', 'Make']].apply(lambda x:x.count())

In [ ]:
#cria tabela que lista o fabricante, o estado e o total de lançamentos, quais tem silicone,
df_b = df_launches.groupby(['KeyManuf','Endereco','Estado'], dropna=False).agg({
    'KeyMarca': to_set,
    'KeyEmpresa': to_set,
    'Total Lançamentos': 'count',
    'Silicone': 'count',
    'Lançamentos': to_set,
    'Ing': sinaliza_ingredientes
})

In [ ]:
#Une tabelas anteriores
df_launches_fab = pd.concat([df_c, df_b], axis=1).reset_index()

In [ ]:
df_launches_fab.to_excel('Relatório Lançamentos.xlsx')
df_launches_fab.head()

,KeyManuf,Endereco,Estado,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing
0,& business,Itália,NaN,0,2,0,"[alfaparf lisse design, alfaparf semi di]",[& business],2,2,[GB 1020],"Dimethicone, Dimethicone/Vinyl Dimethicone Cro..."
1,&co,Brasil,SP,4,0,2,"[pinkcheeks, pinkcheeks pro stick, pinkcheeks ...",[&co],6,5,"[ES 3007, TMS 803]","C30-45 Alkyl Methicone, Cyclopentasiloxane, Di..."
2,21 cosmetics,China,NaN,0,0,2,[la girl],[21 cosmetics],2,2,[],"Caprylyl Methicone, Dimethicone, Silica"
3,3fa,Brasil,GO,0,2,0,"[we pink extraordinary, we pink red]",[3fa],2,2,[GB 1020],"Dimethicone, Dimethiconol"
4,5 cinco quimicos,Brasil,PR,2,0,0,[5 cinco],[5 cinco quimicos],2,0,[],


## Agrupar relatório Vendas e Projetos

In [ ]:
#agrupa lançamentos com vendas e projetos
df_launches_vend_proj = df_launches_fab.merge(df_final, on=['KeyManuf', 'Endereco','Estado'], how='outer')

In [ ]:
df_launches_vend_proj['Tem_Lancamento'] = ~df_launches_vend_proj['Total Lançamentos'].isna()
df_launches_vend_proj['Tem_Venda'] = (
    ~df_launches_vend_proj['Volume_2025'].isna() &
    ~df_launches_vend_proj['Volume_2026'].isna()
)

In [ ]:
df_launches_vend_proj.head()

,KeyManuf,Endereco,Estado,Pele,Cabelos,Make,KeyMarca,KeyEmpresa,Total Lançamentos,Silicone,Lançamentos,Ing,Produtos_2025,Produtos_2026,Volume_2025,Volume_2026,Itens_Adicionados,Itens_Perdidos,Itens_Vendidos_2025_2026,Tem_Lancamento,Tem_Venda
0,& business,Itália,NaN,0.0,2.0,0.0,"[alfaparf lisse design, alfaparf semi di]",[& business],2.0,2.0,[GB 1020],"Dimethicone, Dimethicone/Vinyl Dimethicone Cro...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False
1,&co,Brasil,SP,4.0,0.0,2.0,"[pinkcheeks, pinkcheeks pro stick, pinkcheeks ...",[&co],6.0,5.0,"[ES 3007, TMS 803]","C30-45 Alkyl Methicone, Cyclopentasiloxane, Di...",3.0,2.0,419.5,36.0,[],[DM 5],"[DM 5, EG 6000, TMS 803]",True,True
2,21 cosmetics,China,NaN,0.0,0.0,2.0,[la girl],[21 cosmetics],2.0,2.0,[],"Caprylyl Methicone, Dimethicone, Silica",NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False
3,3fa,Brasil,GO,0.0,2.0,0.0,"[we pink extraordinary, we pink red]",[3fa],2.0,2.0,[GB 1020],"Dimethicone, Dimethiconol",3.0,2.0,581.0,1440.0,[],[EG 5],"[EG 5, GB 1020, OW 2100]",True,True
4,5 cinco quimicos,Brasil,PR,2.0,0.0,0.0,[5 cinco],[5 cinco quimicos],2.0,0.0,[],,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False


# Generate a Report will all information

In [ ]:
df_launches_vend_proj.to_excel('Final.xlsx')